Hourly consumption golden table

In [0]:
from pyspark.sql import functions as F

In [0]:
silver_df = spark.table("energy.silver.consumption")

In [0]:
hourly_df = (
    silver_df
    .withColumn(
        "hour",
        F.date_trunc("hour", "start_time")
    )
    .groupBy("hour")
    .agg(
        F.avg("consumption_mw")
            .alias("avg_consumption_mw"),

        F.min("consumption_mw")
            .alias("min_consumption_mw"),

        F.max("consumption_mw")
            .alias("max_consumption_mw"),

        F.count("*")
            .alias("measurement_count")
    )
)

In [0]:
display(hourly_df)

In [0]:
hourly_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "energy.gold.hourly_consumption"
    )

Daily consumption golden table

In [0]:
daily_df = (
    silver_df
    .withColumn(
        "date",
        F.to_date("start_time")
    )
    .groupBy("date")
    .agg(
        F.avg("consumption_mw")
            .alias("avg_consumption_mw"),

        F.min("consumption_mw")
            .alias("min_consumption_mw"),

        F.max("consumption_mw")
            .alias("max_consumption_mw"),

        F.count("*")
            .alias("measurement_count")
    )
)

In [0]:
daily_df = (
    daily_df
    .withColumn(
        "expected_measurement_count",
        F.lit(480)
    )
    .withColumn(
        "completeness_pct",
        F.round(
            F.col("measurement_count")
            / F.col("expected_measurement_count")
            * 100,
            2
        )
    )
)

In [0]:
display(daily_df)

In [0]:
daily_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "energy.gold.daily_consumption"
    )